In [ ]:
import pandas as pd
import json
import datetime

class AffluenceEstimator:
    def __init__(self, csv_path):
        # Chargement du dataset PRIM (on optimise en typant id_zdc en string)
        print("Chargement des données de référence...")
        self.df = pd.read_csv(csv_path, sep=';')
        self.df['id_zdc'] = self.df['id_zdc'].fillna(0).astype(int).astype(str)

    def _get_cat_jour(self, dt):
        """Détermine la catégorie de jour selon la date."""
        # Note : On pourrait affiner ici en ajoutant un calendrier des vacances scolaires
        weekday = dt.weekday()
        if weekday < 5:
            return 'JOHV'  # Jour Ouvré Hors Vacances
        elif weekday == 5:
            return 'SAHV'  # Samedi Hors Vacances
        else:
            return 'DIJFP' # Dimanche et Fêtes

    def _get_time_slot(self, dt):
        """Formate l'heure au format '17H-18H' du CSV."""
        hour = dt.hour
        next_hour = (hour + 1) % 24
        return f"{hour}H-{next_hour}H"

    def _get_confort_level(self, score):
        """Attribue un libellé de confort selon le pourcentage de validation."""
        if score == 0: return "Inconnu"
        if score < 4: return "Très Fluide (Assis garanti)"
        if score < 7: return "Modéré (Peu de places assises)"
        if score < 10: return "Chargé (Voyage debout probable)"
        return "Saturation (Heure de pointe critique)"

    def process_navitia_json(self, navitia_data):
        """Analyse chaque trajet du JSON Navitia et ajoute les données d'affluence."""
        enriched_journeys = []

        for journey in navitia_data.get('journeys', []):
            journey_report = {
                'duration': journey['duration'],
                'departure': journey['departure_date_time'],
                'sections': [],
                'max_score': 0
            }

            for section in journey.get('sections', []):
                # On ne traite que les sections de transport public
                if section['type'] == 'public_transport':
                    # 1. Extraction du temps
                    dt = datetime.datetime.strptime(section['departure_date_time'], '%Y%m%dT%H%M%S')
                    cat_jour = self._get_cat_jour(dt)
                    time_slot = self._get_time_slot(dt)

                    # 2. Extraction de l'ID station (Stop Area IDFM)
                    # Navitia stocke l'ID sous la forme "stop_area:IDFM:69622"
                    from_obj = section.get('from', {})
                    id_idfm = None

                    # On cherche l'ID dans l'objet stop_area imbriqué
                    if 'stop_point' in from_obj and 'stop_area' in from_obj['stop_point']:
                        id_idfm = from_obj['stop_point']['stop_area']['id'].split(':')[-1]
                    elif 'stop_area' in from_obj:
                        id_idfm = from_obj['stop_area']['id'].split(':')[-1]

                    # 3. Recherche de l'affluence dans le DataFrame
                    score = 0
                    if id_idfm:
                        match = self.df[
                            (self.df['id_zdc'] == id_idfm) &
                            (self.df['cat_jour'] == cat_jour) &
                            (self.df['trnc_horr_60'] == time_slot)
                        ]
                        if not match.empty:
                            score = match.iloc[0]['pourcentage_validations']

                    # 4. Construction du compte-rendu de section
                    sec_info = {
                        'ligne': section.get('display_informations', {}).get('label', '?'),
                        'direction': section.get('display_informations', {}).get('direction', '?'),
                        'depart': section['from']['name'],
                        'arrivee': section['to']['name'],
                        'heure': time_slot,
                        'score_affluence': score,
                        'niveau_confort': self._get_confort_level(score)
                    }
                    journey_report['sections'].append(sec_info)
                    journey_report['max_score'] = max(journey_report['max_score'], score)

            # Qualification globale du trajet
            journey_report['bilan_affluence'] = self._get_confort_level(journey_report['max_score'])
            enriched_journeys.append(journey_report)

        return enriched_journeys

# --- EXEMPLE D'UTILISATION ---
if __name__ == "__main__":
    # 1. Initialiser l'estimateur avec le CSV
    # Remplacez par le bon chemin de fichier
    estimator = AffluenceEstimator('validations-reseau-ferre-profils-horaires-par-jour-type-3eme-trimestre.csv')

    # 2. Charger la réponse Navitia
    with open('response.json', 'r', encoding='utf-8') as f:
        data_navitia = json.load(f)

    # 3. Calculer l'affluence
    resultats = estimator.process_navitia_json(data_navitia)

    # 4. Affichage des résultats
    print("\n--- ANALYSE DES ITINÉRAIRES ---")
    for i, res in enumerate(resultats):
        print(f"\nOption {i+1} ({res['duration']//60} min) :")
        print(f"Bilan global : {res['bilan_affluence']}")
        for s in res['sections']:
            print(f"  [{s['ligne']}] {s['depart']} -> {s['arrivee']}")
            print(f"      Affluence : {s['score_affluence']}% ({s['niveau_confort']})")

Chargement des données de référence...

--- ANALYSE DES ITINÉRAIRES ---

Option 1 (44 min) :
Bilan global : Saturation (Heure de pointe critique)
  [B] Les Baconnets (Antony) -> Cité Universitaire (Paris)
      Affluence : 6.69% (Modéré (Peu de places assises))
  [T3a] Cité Universitaire (Paris) -> Porte de Choisy (Paris)
      Affluence : 10.05% (Saturation (Heure de pointe critique))
  [7] Porte de Choisy (Paris) -> Mairie d'Ivry (Ivry-sur-Seine)
      Affluence : 5.8% (Modéré (Peu de places assises))

Option 2 (47 min) :
Bilan global : Chargé (Voyage debout probable)
  [B] Les Baconnets (Antony) -> Denfert-Rochereau (Paris)
      Affluence : 6.69% (Modéré (Peu de places assises))
  [6] Denfert-Rochereau (Paris) -> Place d'Italie (Paris)
      Affluence : 9.38% (Chargé (Voyage debout probable))
  [7] Place d'Italie (Paris) -> Mairie d'Ivry (Ivry-sur-Seine)
      Affluence : 9.76% (Chargé (Voyage debout probable))

Option 3 (57 min) :
Bilan global : Saturation (Heure de pointe critiqu